# Atmosphere & chemistry — Sentinel-5P NO2

Tropospheric NO2 column density from TROPOMI on Sentinel-5P (near-real-time, ~7 km native pixel). A short window over Paris, reduced to one composite scene.

## Setup

`pyramids` provides `Dataset` (GeoTIFF/NetCDF reading + plotting); `earthlens` provides the
unified `EarthLens` entry point and the GEE `Catalog`. We also create a per-notebook `out/`
directory for the downloaded GeoTIFFs.

In [ ]:
import os
from pathlib import Path

from pyramids.dataset import Dataset as PyramidsDataset

from earthlens.core import EarthLens
from earthlens.gee import Catalog

OUT_DIR = Path('out') / 'atmosphere-chemistry'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'output directory: {OUT_DIR.resolve()}')

### Credentials

The notebook reads the GEE service-account credentials from the `GEE_SERVICE_ACCOUNT` /
`GEE_SERVICE_KEY` environment variables. Both must be set before running this cell.

In [ ]:
SERVICE_ACCOUNT = os.environ['GEE_SERVICE_ACCOUNT']
SERVICE_KEY = os.environ['GEE_SERVICE_KEY']

## Inspect the catalog entry

Before downloading anything, look at what the bundled catalog knows about the asset — bands, cadence, license, provider.

In [ ]:
cat = Catalog()
ds = cat.get_dataset('COPERNICUS/S5P/NRTI/L3_NO2')
print(f'title:               {ds.title}')
print(f'ee_type:             {ds.ee_type}')
print(f'spatial_resolution:  {ds.spatial_resolution} m')
print(f'extent.start_date:   {ds.extent.start_date}')
print(f'extent.end_date:     {ds.extent.end_date}')
print(f'default_reducer:     {ds.default_reducer}')
print(f'license:             {ds.license}')
print(f'provider:            {ds.provider}')
print(f'#bands:              {len(ds.bands)}')
print(f'band ids (first 5):  {list(ds.bands)[:5]}')

## Download

Tiny AOI ([48.5, 49.5] lat, [2.0, 3.0] lon) at 7000.0 m, `raw` cadence — keeps the synchronous download under EE's 32768-px per-axis cap.

### Build the request and authenticate

Construct the `EarthLens` request first, then resolve the GEE service-account credentials with
`authenticate()` on its own line so each step is easy to read and re-run.

In [ ]:
gee = EarthLens(
    data_source="gee",
    start='2024-01-05',
    end='2024-01-10',
    dataset='COPERNICUS/S5P/NRTI/L3_NO2',
    variables=['tropospheric_NO2_column_number_density'],
    aoi=[2.0, 48.5, 3.0, 49.5],
    cadence='raw',
    path=str(OUT_DIR),
    scale=7000.0,
    reducer='mean',
)
gee.authenticate(service_account=SERVICE_ACCOUNT, service_key=SERVICE_KEY)

### Fetch the composite

`download()` writes the reduced scene to disk and returns the list of GeoTIFF paths. The
`try/except` keeps the graceful-skip guard intact for environments where the live EE call cannot
run.

In [ ]:
try:
    paths = gee.download(progress_bar=False)
    print(f'wrote {len(paths)} GeoTIFF(s):')
    for p in paths:
        print(f'  {p}  ({p.stat().st_size / 1024:.1f} KB)')
    download_ok = True
except Exception as exc:
    if False:
        print(
            f'live EE call failed (tolerated for this category): {type(exc).__name__}: {exc}'
        )
        paths = []
        download_ok = False
    else:
        raise

## Quick preview

Load the first written GeoTIFF through pyramids and render the single band. (`pyramids.dataset.Dataset` is the project's GeoTIFF/NetCDF wrapper.)

### Load and mask the band

Read the first written GeoTIFF through pyramids, collapse it to a single band, and mask the
dataset's nodata value so the colormap isn't pinned to it.

In [ ]:
if not download_ok or not paths:
    print('Skipping preview — no GeoTIFF was written.')
    preview = None
else:
    # plot() resolves the band and honours its declared no-data, so the array
    # is never read, sliced or masked by hand here.
    preview = PyramidsDataset.read_file(paths[0])

### Render the NO2 column

Plot the masked band with a `viridis` colormap and print the value range so the units are
concrete.

In [ ]:
if preview is not None:
    preview.plot(
        cmap='viridis',
        title=f"{'COPERNICUS/S5P/NRTI/L3_NO2'} / {'tropospheric_NO2_column_number_density'}",
    )
    # stats() reports the band's real range with no-data already excluded,
    # which is what the hand-masked nanmin / nanmax calls stood in for.
    stats = preview.stats(approx_ok=False)
    low, high = stats['min'].iloc[0], stats['max'].iloc[0]
    print(f'value range: [{low:.4g}, {high:.4g}]')

## Tracking submitted jobs (asynchronous export)

The download above uses `export_via="url"` — a synchronous `getDownloadURL` round-trip. Nothing was queued, so there's no Earth Engine job to track.

To track an export instead, switch to an asynchronous sink (`drive` / `gcs` / `asset`) and pass `wait_for_export=False` so `.download()` returns a `TaskInfo` at submission time rather than blocking until completion. The cells below submit the same `(asset_id, band, AOI, scale)` request as an `export_via="asset"` task into the service account's own asset folder, then walk the four jobs-API calls (`list_recent_tasks` → `wait_for_task_id` → `ee.data.getAsset` → `ee.data.deleteAsset`) to make the job finish *and* tidy up. See `track-batch-exports.ipynb` for a deeper worked example.

### Imports and the demo asset folder

Bring in the jobs-API helpers and derive a `Folder` asset path under the current project. The
backend writes the image at `<asset_id>/<prefix>`, so this `asset_id` is the parent folder, not
the final image path.

In [ ]:
import ee

from earthlens.gee import cancel_task, list_recent_tasks, wait_for_task_id

# The asset goes into a `Folder` asset that we own. `GEE._export_via_batch`
# writes the actual image at `<asset_id>/<prefix>`, so `asset_id` here is
# the parent FOLDER (not the final image path). Both must be cleaned up.
_proj = ee.data._get_projects_path().removeprefix('projects/')
DEMO_FOLDER = f'projects/{_proj}/assets/earthlens-demo-atmosphere-chemistry'
print(f'demo folder: {DEMO_FOLDER}')

### Prepare a clean folder

Best-effort tear down any leftovers from a previous run, then (re)create the parent folder —
Earth Engine requires it to exist before a child write.

In [ ]:
# Best-effort cleanup of leftover children from a previous run (so the
# folder is empty before we try to delete it below).
try:
    for child in ee.data.listAssets({'parent': DEMO_FOLDER}).get('assets', []):
        ee.data.deleteAsset(child['name'])
        print(f'cleared leftover child: {child["name"]}')
    ee.data.deleteAsset(DEMO_FOLDER)
    print(f'cleared leftover folder: {DEMO_FOLDER}')
except Exception:
    pass
# Create the parent folder — EE requires it to exist before a child write.
ee.data.createAsset({'type': 'Folder'}, DEMO_FOLDER)
print(f'created folder: {DEMO_FOLDER}')

### Submit

Same `(asset_id, band, AOI, scale)` request as the sync download above, just routed through `export_via="asset"` + `wait_for_export=False`. `download()` returns a `TaskInfo` per submitted bucket at the moment the task is queued — no blocking.

### Build the async request and authenticate

Same `(asset_id, band, AOI, scale)` request as the sync download, but routed through
`export_via="asset"` + `wait_for_export=False`. Construct it first, then `authenticate()` on its
own line.

In [ ]:
async_gee = EarthLens(
    data_source="gee",
    start='2024-01-05',
    end='2024-01-10',
    dataset='COPERNICUS/S5P/NRTI/L3_NO2',
    variables=['tropospheric_NO2_column_number_density'],
    aoi=[2.0, 48.5, 3.0, 49.5],
    cadence='raw',
    path=str(OUT_DIR),
    scale=7000.0,
    reducer='mean',
    export_via='asset',
    asset_id=DEMO_FOLDER,
    wait_for_export=False,
)
async_gee.authenticate(service_account=SERVICE_ACCOUNT, service_key=SERVICE_KEY)

### Submit the export

`download()` returns a `TaskInfo` per submitted bucket at the moment the task is queued — no
blocking. The `try/except` keeps the graceful-skip guard.

In [ ]:
submitted_ok = False
task_info = None
try:
    submitted = async_gee.download(progress_bar=False)
    task_info = submitted[0]
    submitted_ok = True
    print(f'submitted: id={task_info.id} state={task_info.state}')
    print(f'           description={task_info.description}')
except Exception as exc:
    if False:
        print(f'async submission failed (tolerated): {type(exc).__name__}: {exc}')
    else:
        raise

### List + wait

`list_recent_tasks(description_prefix=...)` returns every matching task across the current project; `wait_for_task_id` blocks until the one we care about reaches a terminal state. A real workflow would just poll later from a separate process — the wait here exists so the notebook shows the full success path end-to-end.

In [ ]:
if submitted_ok and task_info is not None:
    recent = list_recent_tasks(
        description_prefix=task_info.description,
        max_age_min=10,
    )
    print(f'list_recent_tasks matched {len(recent)} task(s):')
    for t in recent:
        print(f'  {t.id}  {t.state:<12} {t.description}')
    try:
        final = wait_for_task_id(
            task_info.id,
            poll_seconds=10,
            progress_bar=False,
        )
        print(f'\nfinal state: {final.state}')
    except RuntimeError as exc:
        # Raised on FAILED / CANCELLED — cancel-if-still-running
        # so we don't leak an in-flight task on notebook restart.
        print(f'wait_for_task_id raised: {exc}')
        try:
            cancel_task(task_info.id)
        except Exception:
            pass
else:
    print('Skipping list/wait — async submission did not succeed.')

### Verify + clean up

Confirm the produced asset exists on Earth Engine, then delete it (and the surrounding demo folder) so we don't leak storage between notebook runs. The backend wrote the image at `<DEMO_FOLDER>/<task description>`.

In [ ]:
if submitted_ok and task_info is not None:
    produced = f'{DEMO_FOLDER}/{task_info.description}'
    try:
        meta = ee.data.getAsset(produced)
        print(f'asset exists: type={meta.get("type")} name={meta.get("name")}')
        ee.data.deleteAsset(produced)
        print('asset deleted')
    except Exception as exc:
        print(f'verify/delete skipped: {exc}')
# Always try to tear down the parent folder.
try:
    ee.data.deleteAsset(DEMO_FOLDER)
    print(f'folder deleted: {DEMO_FOLDER}')
except Exception as exc:
    print(f'folder delete skipped: {exc}')

## What's on disk

The GeoTIFF (or empty list, if the EE call was tolerated) is left under the per-notebook `out/` directory for you to inspect. That directory is `.gitignore`d — re-running the notebook overwrites it.

In [ ]:
for p in sorted(OUT_DIR.iterdir()) if OUT_DIR.exists() else []:
    print(f'{p}  ({p.stat().st_size / 1024:.1f} KB)')